# Official TENTACLES preprocessing — executable smoke test

Pinned original R package, synthetic RNA-seq counts, direct `preProcess()` execution. This is a component test, not reproduction of the complete published analysis.

In [4]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import shutil
import subprocess

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    pass

paths = [Path("/content/drive/MyDrive/CMPS355_Research"), Path.cwd(), *Path.cwd().parents]
ROOT = next((path for path in paths if (path / "configs" / "dataset.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("CMPS355_Research was not found")
SCRIPT = ROOT / "r" / "test_tentacles_preprocess.R"
if not SCRIPT.is_file():
    raise FileNotFoundError(SCRIPT)
RSCRIPT = shutil.which("Rscript")
if not RSCRIPT:
    raise RuntimeError("Rscript is missing from this Colab runtime")
SHA = "069a26271e3885e015df25baa40a7a1140827d5f"
LIB = Path("/content/cmps355_r_lib")
RUNS = ROOT / "runs" / "official_r"
RUNS.mkdir(parents=True, exist_ok=True)
print("Project:", ROOT)
print("R library:", LIB)
print("R package cache available:", LIB.is_dir())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project: /content/drive/MyDrive/CMPS355_Research
R library: /content/cmps355_r_lib
R package cache available: False


## R environment restoration


In [5]:
import platform
import tarfile
import tempfile

version = subprocess.check_output(
    [RSCRIPT, "--vanilla", "-e", "cat(as.character(getRversion()))"], text=True
).strip()
os_info = platform.freedesktop_os_release()
key = f"r{version}_{os_info.get('ID', 'linux')}{os_info.get('VERSION_ID', 'unknown')}_{platform.machine()}"
cache_dir = ROOT / "runs" / "environment_cache"
archive = cache_dir / f"{key}.tar.gz"
manifest_path = cache_dir / f"{key}.json"

def package_revision():
    environment = os.environ.copy()
    environment["CMPS355_R_LIB"] = str(LIB)
    expression = (
        'lib <- Sys.getenv("CMPS355_R_LIB"); '
        'if (dir.exists(lib)) .libPaths(c(lib, .libPaths())); '
        'if (!requireNamespace("TENTACLES", quietly=TRUE)) quit(status=1L); '
        'd <- utils::packageDescription("TENTACLES"); '
        'cat(tolower(d[["RemoteSha"]]))'
    )
    result = subprocess.run(
        [RSCRIPT, "--vanilla", "-e", expression],
        env=environment, capture_output=True, text=True
    )
    return result.stdout.strip() if result.returncode == 0 else None

revision = package_revision()
if revision != SHA:
    if not archive.is_file() or not manifest_path.is_file():
        available = sorted(path.name for path in cache_dir.glob("*.json")) if cache_dir.is_dir() else []
        raise RuntimeError(
            f"TENTACLES revision {SHA} is not installed and a compatible Drive cache was not found. "
            f"Expected: {archive}. Available manifests: {available}. "
            "The cached installer notebook 04 must be run to restore or build the environment."
        )
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    expected = {
        "r_version": version,
        "os": os_info.get("ID", "linux"),
        "os_version": os_info.get("VERSION_ID", "unknown"),
        "arch": platform.machine(),
    }
    for name, value in expected.items():
        if manifest.get(name) != value:
            raise RuntimeError(f"R cache incompatibility for {name}: {manifest.get(name)} != {value}")
    with archive.open("rb") as source:
        digest = hashlib.file_digest(source, "sha256").hexdigest()
    if digest != manifest.get("sha256"):
        raise RuntimeError("R environment cache checksum mismatch")
    LIB.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(dir="/content" if Path("/content").is_dir() else None) as location:
        with tarfile.open(archive, "r:gz") as bundle:
            members = bundle.getmembers()
            for member in members:
                parts = Path(member.name).parts
                if not parts or parts[0] != "r_library" or ".." in parts:
                    raise RuntimeError("Unsafe archive member")
            bundle.extractall(location, filter="data")
        restored = Path(location) / "r_library"
        if not restored.is_dir():
            raise RuntimeError("Cache archive does not contain the expected R library")
        count = 0
        for package in restored.iterdir():
            destination = LIB / package.name
            if package.is_dir() and not destination.exists():
                shutil.copytree(package, destination)
                count += 1
    print("Restored R packages:", count)
    revision = package_revision()

if revision != SHA:
    raise RuntimeError(f"TENTACLES revision mismatch or unable to load: {revision!r}")
print("TENTACLES revision verified:", revision)


Restored R packages: 140
TENTACLES revision verified: 069a26271e3885e015df25baa40a7a1140827d5f


## Direct author-package execution

In [7]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
log_file = RUNS / f"tentacles_preprocess_{stamp}.log"
env = os.environ.copy()
env["CMPS355_R_LIB"] = str(LIB)
with log_file.open("w", encoding="utf-8") as stream:
    try:
        proc = subprocess.run(
            [RSCRIPT, str(SCRIPT), SHA],
            stdout=stream, stderr=subprocess.STDOUT,
            env=env, timeout=300, check=False
        )
        exit_code = proc.returncode
    except subprocess.TimeoutExpired:
        exit_code = -1
        stream.write("\nTIMEOUT\n")

log_lines = log_file.read_text(encoding="utf-8", errors="replace").splitlines()
source_confirmed = f"SOURCE_SHA={SHA}" in log_lines
executed = "PREPROCESS_EXECUTED=TRUE" in log_lines
non_null = "PREPROCESS_NON_NULL=TRUE" in log_lines
passed = exit_code == 0 and source_confirmed and executed and non_null
status = "PASS_PREPROCESS_EXECUTED_SYNTHETIC" if passed else "FAIL_PREPROCESS_EXECUTION"
report = {
    "method": "TENTACLES", "stage": "official_preprocess_component",
    "status": status, "source_commit": SHA,
    "r_script_sha256": hashlib.sha256(SCRIPT.read_bytes()).hexdigest(),
    "run_exit_code": exit_code,
    "source_confirmed": source_confirmed,
    "preprocess_executed": executed,
    "preprocess_returned_non_null": non_null,
    "log": str(log_file.relative_to(ROOT)),
    "scope": "synthetic_preprocessing_only_not_full_paper",
    "created_utc": datetime.now(timezone.utc).isoformat()
}
report_file = RUNS / f"tentacles_preprocess_report_{stamp}.json"
report_file.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print("\n".join(log_lines[-35:]))
print("Report:", report_file.relative_to(ROOT))
print("Status:", status)
if not passed:
    raise RuntimeError("Official preprocessing check failed; inspect the log")

 [34] R6_2.6.1              stringi_1.8.9         rsample_1.3.2        
 [37] RColorBrewer_1.1-3    limma_3.68.5          genefilter_1.94.0    
 [40] rules_1.0.3           parallelly_1.50.0     rpart_4.1.27         
 [43] lubridate_1.9.5       Seqinfo_1.2.0         Rcpp_1.1.2           
 [46] assertthat_0.2.1      dials_1.4.4           iterators_1.0.14     
 [49] future.apply_1.21.0   zoo_1.9-1             IRanges_2.46.0       
 [52] Matrix_1.7-6          splines_4.6.1         nnet_7.3-21          
 [55] timechange_0.4.0      tidyselect_1.2.1      viridis_0.6.5        
 [58] partykit_1.3-1        TSP_1.2.7             timeDate_4052.112    
 [61] codetools_0.2-20      listenv_1.1.0         lattice_0.23-1       
 [64] tibble_3.3.1          plyr_1.8.9            KEGGREST_1.52.2      
 [67] Biobase_2.72.0        withr_3.0.3           S7_0.2.2             
 [70] future_1.82.0         survival_3.8-12       heatmaply_1.6.0      
 [73] Biostrings_2.80.2     mclust_6.1.3          pillar_1.11.1 